In [5]:
from ortools.linear_solver import pywraplp

# 1. ENTRADA DE DADOS GENÉRICA
print("--- Configuração do Problema ---")
L = int(input("Tamanho da barra (Ex: 150): "))
num_tipos = int(input("Quantidade de tipos de itens (Ex: 3): "))

itens = []
demandas = []
for i in range(num_tipos):
    tamanho = int(input(f"Tamanho do item {i+1}: "))
    qtd = int(input(f"Demanda do item {i+1}: "))
    itens.append(tamanho)
    demandas.append(qtd)

--- Configuração do Problema ---


KeyboardInterrupt: Interrupted by user

In [ ]:
# 2. GERAÇÃO DE PADRÕES DE CORTE (Pre-processing)
# Gera todos os padrões "maximais" (onde não cabe mais nenhum item do estoque no espaço restante)
padroes = []

def gerar_padroes(idx, gasto_atual, padrao_atual):
    if idx == num_tipos:
        # Verifica se o padrão é maximal (eficiente)
        pode_adicionar_mais = False
        for item in itens:
            if gasto_atual + item <= L:
                pode_adicionar_mais = True
                break

        if not pode_adicionar_mais and sum(padrao_atual) > 0:
            padroes.append(list(padrao_atual))
        return

    # Quantidade máxima que o item atual pode aparecer
    max_qtd = (L - gasto_atual) // itens[idx]
    for qtd in range(int(max_qtd) + 1):
        padrao_atual[idx] = qtd
        gerar_padroes(idx + 1, gasto_atual + qtd * itens[idx], padrao_atual)

print("Gerando padrões de corte válidos...")
gerar_padroes(0, 0, [0] * num_tipos)
print(f"Total de padrões gerados: {len(padroes)}")

In [ ]:
# 3. CRIAÇÃO DO SOLVER E MODELAGEM
solver = pywraplp.Solver.CreateSolver('SCIP')
infinity = solver.infinity()

if not solver:
    print("Erro: Solver SCIP não disponível.")
else:
    # 4. VARIÁVEIS DE DECISÃO
    # x[j] = quantidade de vezes que o padrão j será utilizado
    x = {}
    for j in range(len(padroes)):
        x[j] = solver.IntVar(0, infinity, f'x_{j}')

    # 5. FUNÇÃO OBJETIVO: Minimizar o total de barras brutas utilizadas
    objetivo = solver.Objective()
    for j in range(len(padroes)):
        objetivo.SetCoefficient(x[j], 1)
    objetivo.SetMinimization()

    # 6. RESTRIÇÕES: Garantir que a demanda de cada tipo de item seja atendida
    for i in range(num_tipos):
        # Soma(qtd_item_no_padrao * vezes_que_usa_padrao) >= demanda
        restricao = solver.Constraint(demandas[i], infinity, f'rest_demanda_item_{itens[i]}')
        for j in range(len(padroes)):
            if padroes[j][i] > 0:
                restricao.SetCoefficient(x[j], padroes[j][i])

    # 7. RESOLUÇÃO
    status = solver.Solve()

In [ ]:
# 8. EXIBIÇÃO DOS RESULTADOS
if status == pywraplp.Solver.OPTIMAL:
    total_barras = int(solver.Objective().Value())
    print(f"\n{'='*30}")
    print(f"SOLUÇÃO ÓTIMA ENCONTRADA")
    print(f"{'='*30}")
    print(f"Total de barras de {L}m utilizadas: {total_barras}")

    # Cálculo do Desperdício (Total comprado - Total entregue)
    comprimento_util = sum(itens[i] * demandas[i] for i in range(num_tipos))
    desperdicio_total = (total_barras * L) - comprimento_util
    print(f"Desperdício total: {desperdicio_total}m")

    print("\nPadrões de corte ativos:")
    for j in range(len(padroes)):
        qtd_uso = int(x[j].solution_value())
        if qtd_uso > 0:
            detalhe_corte = ", ".join([f"{padroes[j][k]}x({itens[k]}m)" for k in range(num_tipos) if padroes[j][k] > 0])
            print(f"- Padrão {j} [{detalhe_corte}]: usado {qtd_uso} vezes")

    # 9. EXPORTAÇÃO DO MODELO LP (Exigência do trabalho)
    print("\n--- MODELO MATEMÁTICO (LP FORMAT) ---")
    print(solver.ExportModelAsLpFormat(False))
else:
    print("O solver não conseguiu encontrar uma solução ótima.")
